In [1]:
# Define Checkpoint Path (Set this to your specific checkpoint directory)
CHECKPOINT_PATH = "/home/new/Documents/QSH/Llama_train/outputs/Qwen3_02-15/method_FSL_bs_2_inputdata_input_text_Qwen4B_FSL32/checkpoints/checkpoint-epoch-4" # Example path
LOAD_FROM_CHECKPOINT = True # Set to False to load base model only

# Generation Task Debugging & Analysis Pipeline

## Objective
This notebook is designed to debug the **Few-Shot Learning (FSL) pipeline** for generation tasks.
We will investigate:
1.  **Metric Anomalies**: Why PPL is high (>700) and Diversity (Dist-1/2) is near 1.0.
2.  **Data Loading Issues**: Verify Prompt/Target alignment and Masking.
3.  **Chat Template**: Check if `apply_chat_template` is introducing artifacts.
4.  **Generation Quality**: Test `left-padding` mechanics and model output.

***

## 1. Environment & Config Setup
Initialize `GenTrainingConfig` specifically for the problematic FSL scenario (e.g., 32 shots).

In [2]:
import sys
import os
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.utils.data import DataLoader
nltk_path = os.path.expanduser('~/nltk_data')
import nltk
if not os.path.exists(nltk_path):
    os.makedirs(nltk_path, exist_ok=True)
nltk.data.path.append(nltk_path)
try:
    nltk.data.find('tokenizers/punkt')
except LookupError:
    nltk.download('punkt', download_dir=nltk_path)
    nltk.download('punkt_tab', download_dir=nltk_path)

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from ZGeneration.config_gen import GenTrainingConfig
from ZGeneration.data_loader_gen import load_gen_data, GenerationDataset, gen_loader_warp
from ZGeneration.model_loader_gen import GenModelLoader

# Mock Config mimicking the failure case
class DebugConfig(GenTrainingConfig):
    def __init__(self):
        super().__init__()
        self.few_shot = True
        self.fast_train = False
        self.shots_per_class = 32 # The problematic setting
        self.batch_size = 2
        self.max_seq_length = 2048
        self.model_name = "Qwen3-4B-Instruct-2507" # "Llama-3.3-8B-Instruct" # Or a smaller proxy if weights are huge
        self.hidden_size = 2560
        self.device = "cuda:3" if torch.cuda.is_available() else "cpu"
        self.cuda_device = 3
        self.data_path = os.path.join(project_root, "data")

config = DebugConfig()
print("Config Initialized:", config.__dict__)

/home/new/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Config Initialized: {'model_name': 'Qwen3-4B-Instruct-2507', 'num_emotions': 32, 'hidden_size': 2560, 'data_path': '/home/new/Documents/QSH/Llama_train/data', 'prompt_key': 'input_text', 'dialogue_window': 3, 'max_seq_length': 2048, 'batch_size': 2, 'num_workers': 4, 'topic_name': 'LlamaTest', 'learning_rate': 2e-05, 'num_epochs': 10, 'warmup_steps': 500, 'gradient_accumulation_steps': 2, 'max_grad_norm': 1.0, 'weight_decay': 0.01, 'few_shot': True, 'shots_per_class': 32, 'semi_supervised': True, 'semi_ratio': 0.2, 'fast_train': False, 'quant': False, 'log_interval': 100, 'eval_interval': 1000, 'save_interval': 1000, 'output_dir': './outputs', 'experiment_name': 'Llama_02-22', 'param1': 'method', 'param2': 'bs', 'param3': 'inputdata', 'device': 'cuda:3', 'cuda_device': 3, 'fp16': True, 'seed': 42, 'metrics': ['accuracy', 'f1', 'precision', 'recall'], 'max_new_tokens': 150, 'gen_temperature': 0.7, 'gen_top_p': 0.9, 'eval_bleu': True, 'eval_rouge': True, 'val_ratio': 0.1, 'test_ratio': 0

In [3]:
print(config.cuda_device, config.device)

3 cuda:3


## 2. Load Tokenizer & Inspect Raw Data Correctness
We need to see what the raw data looks like before tokenization.

In [10]:
# Load Tokenizer & Model (Base or Checkpoint)
from peft import PeftModel

# Ensure CUDA device is visible if we want to map correctly
if config.cuda_device is not None:
    # If we are strictly using one GPU, we can set visible devices
    # But usually, if we pass specific device index, torch handles it
    pass

try:
    # 1. Load Base Model & Tokenizer using Loader
    loader = GenModelLoader(config.model_name, config.device)
    tokenizer, model = loader.load() 
    
    print(f"Base Model loaded on {model.device}")

    # 2. Load Trained Checkpoint (Adapter)
    if LOAD_FROM_CHECKPOINT and os.path.exists(CHECKPOINT_PATH):
        print(f"Loading Adapter from Checkpoint: {CHECKPOINT_PATH}")
        
        from peft import PeftModel
        if isinstance(model, PeftModel):
             print("Model is already PEFT. Loading adapter...")
             # Unload execution: To cleanly reload the trained adapter
             base_model = model.unload()
             
             # Reload Adapter
             # Use the same device map as configuration
             model = PeftModel.from_pretrained(
                 base_model, 
                 CHECKPOINT_PATH, 
                 is_trainable=False, 
                 torch_device="cuda:3" # Let accelerate handle it based on base_model placement
             )
             print("Checkpoint Adapter Loaded Successfully.")
        else:
             model = PeftModel.from_pretrained(model, CHECKPOINT_PATH, device_map="auto")

except Exception as e:
    print(f"Error loading model/checkpoint: {e}")
    # Fallback for Tokenizer checks
    tokenizer = AutoTokenizer.from_pretrained(config.model_name, trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

# Load Raw Data
data_dir = os.path.join(config.data_path, 'gen_task')
raw_data, max_len = load_gen_data(data_dir)

Model found locally at: /home/new/Documents/LLModel/Qwen3-4B-Instruct-2507
Loading Model with Quantization: True


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading checkpoint shards: 100%|██████████| 4/4 [00:06<00:00,  1.56s/it]


trainable params: 33,030,144 || all params: 4,054,817,280 || trainable%: 0.8146
Base Model loaded on cuda:3
Loading Adapter from Checkpoint: /home/new/Documents/QSH/Llama_train/outputs/Qwen3_02-15/method_FSL_bs_2_inputdata_input_text_Qwen4B_FSL32/checkpoints/checkpoint-epoch-4
Model is already PEFT. Loading adapter...


/home/new/miniconda3/lib/python3.13/site-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


Checkpoint Adapter Loaded Successfully.
Loading Gen Data: /home/new/Documents/QSH/Llama_train/data/gen_task/processed_gen_data.pkl
Data Loaded: 51239 samples.


In [11]:
base_model.device

device(type='cuda', index=3)

In [12]:
# Inspect one sample
idx = 1
print(f"Sample {idx} Input (Type: {type(raw_data['input_text'][idx])}):")
print(raw_data['input_text'][idx])
print(f"Sample {idx} Target:")
print(raw_data['target_text'][idx])

Sample 1 Input (Type: <class 'list'>):
[{'role': 'system', 'content': "U r an empathetic assistant. You need to understand the user's situation, feelings and respond supportively."}, {'role': 'user', 'content': 'Situation: i remember going to the fireworks with my best friend . there was a lot of people , but it only felt like us in the world .\n\nUser Word: i remember going to see the fireworks with my best friend . it was the first time we ever spent time alone together . although there was a lot of people , we felt like the only people in the world .'}, {'role': 'assistant', 'content': 'was this a friend you were in love with , or just a best friend ?'}, {'role': 'user', 'content': 'this was a best friend . i miss her .'}, {'role': 'assistant', 'content': 'where has she gone ?'}]
Sample 1 Target:
where has she gone ?


In [13]:
print(config.max_seq_length, max_len)
config.max_seq_length = max_len
# config.max_seq_length = 2048

2048 2183


## 3. Deep Dive: Dataset Construction & Masking
Create the `GenerationDataset` and retrieve one item. Check:
- **Left Padding** on `prompt_ids`?
- **Masking (-100)** on `labels`?
- **Alignment**: Does the unmasked part associated with `labels` match the target text?

In [23]:
def label_token_match_check(labels, input_ids, tokenizer):
    for lbl, inids in zip(labels, input_ids):
        if lbl != -100:
            lbl_token = tokenizer.decode([lbl], skip_special_tokens=False)
            inid_token = tokenizer.decode([inids], skip_special_tokens=False)
            # print(f"Label Token: '{lbl_token}' | Input ID Token: '{inid_token}'")
            assert lbl_token == inid_token, f"Label token '{lbl_token}' does not match input ID token '{inid_token}'"

In [14]:
# Create Dataset
ds = GenerationDataset(raw_data, tokenizer, config.max_seq_length)
# Check for overlap or off-by-one
# If decoded labels include part of the prompt, Masking is wrong.

In [ ]:
for i in range(len(ds)):
    item = ds[i]
    try:
        label_token_match_check(item['labels'], item['input_ids'], tokenizer)
    except AssertionError as e:
        print(f"\n--- Sample {i} ---")

In [19]:
for i in range(10):
    item = ds[i]

    # print("Keys:", item.keys())
    # print(f"Input Shape: {item['input_ids'].shape}")
    # print(f"Labels Shape: {item['labels'].shape}")
    # print(f"Prompt IDs Shape: {item['prompt_ids'].shape}")

    # # Visual Inspection of Alignment
    # # Decode the Prompt
    # decoded_prompt = tokenizer.decode(item['prompt_ids'], skip_special_tokens=False)
    # print("\n--- Decoded Prompt (Left Padded?) ---")
    # print(decoded_prompt) # Show last 500 chars

    # Decode the Target region (Labels != -100)
    valid_label_indices = item['labels'] != -100
    target_tokens = item['labels'][valid_label_indices][:-2]
    decoded_target_from_labels = tokenizer.decode(target_tokens, skip_special_tokens=False)
    print("\n--- Decoded Labels (The Training Target) ---")
    print(decoded_target_from_labels)

    print("\n--- Original Target Text ---")
    print(item['target_text'])

    print(len(decoded_target_from_labels), len(item['target_text']))


--- Decoded Labels (The Training Target) ---
was this a friend you were in love with , or just a best friend ?

--- Original Target Text ---
was this a friend you were in love with , or just a best friend ?
65 65

--- Decoded Labels (The Training Target) ---
where has she gone ?

--- Original Target Text ---
where has she gone ?
20 20

--- Decoded Labels (The Training Target) ---
oh was this something that happened because of an argument ?

--- Original Target Text ---
oh was this something that happened because of an argument ?
60 60

--- Decoded Labels (The Training Target) ---
oh ya ? i do not really see how

--- Original Target Text ---
oh ya ? i do not really see how
31 31

--- Decoded Labels (The Training Target) ---
i do actually hit blank walls a lot of times but i get by

--- Original Target Text ---
i do actually hit blank walls a lot of times but i get by
57 57

--- Decoded Labels (The Training Target) ---
wait what are sweatings

--- Original Target Text ---
wait what are 

## 4. Metric Validation (The "Dist-1=1.0" Bug)
Simulate the bug by calculating Dist-1/Dist-2 on a single short sentence vs. a corpus.

In [ ]:
from nltk.translate.bleu_score import sentence_bleu

def broken_metric_logic(preds):
    """Replicates the per-sample logic found in train_gen.py"""
    d1_scores = []
    for p in preds:
        toks = nltk.word_tokenize(p)
        if len(toks) == 0: continue
        d1 = len(set(toks)) / len(toks)
        d1_scores.append(d1)
    return np.mean(d1_scores)

def correct_metric_logic(preds):
    """Calculates diversity over the whole corpus (or concatenated)"""
    all_toks = []
    for p in preds:
        all_toks.extend(nltk.word_tokenize(p))
    if not all_toks: return 0.0
    return len(set(all_toks)) / len(all_toks)

# Test Case
samples = ["I am happy", "You are happy", "We are valid"]
print(f"Samples: {samples}")
print(f"Broken Logic (Per-Sample Avg): {broken_metric_logic(samples)} (Expect ~1.0 if short)")
print(f"Correct Logic (Corpus Level): {correct_metric_logic(samples)} (Expect < 1.0 due to 'happy', 'are' repeats)")

## 5. Generation Test (Left Padding Check)
Perform a real generation call. If `left-padding` is broken, the model sees PADS as the most recent tokens and generates nonsense.

In [ ]:
if model:
    model.eval()
    
    b_input = item['prompt_ids'].unsqueeze(0).to(config.device)
    b_mask = item['prompt_mask'].unsqueeze(0).to(config.device)
    
    print("\nGeneratng...")
    with torch.no_grad():
        gen_out = model.generate(
            b_input, 
            attention_mask=b_mask,
            max_new_tokens=50,
            do_sample=True,
            temperature=0.7,
            pad_token_id=tokenizer.pad_token_id
        )
    
    # Decode
    # remove prompt part
    new_tokens = gen_out[0][b_input.shape[1]:]
    out_text = tokenizer.decode(new_tokens, skip_special_tokens=True)
    print(f"Generated: {out_text}\nTarget Text: {item['target_text']}")
    
    # Check if it's nonsense
else:
    print("Model not loaded, skipping generation test.")


Generatng...
Generated: yeah , i can imagine .
Target Text: what do you mean it has n't been easy ? how close have you come to cheating ?
